# FASTT — main results on LibriSpeech

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/fastt/blob/main/notebooks/main_librispeech.ipynb)

This notebook reproduces the paper's main experiment on LibriSpeech. Every method's hyperparameters are selected on
a 25 % hold-out of the corpus; every method is then trained and tested on the same ten folds of a 5×2
cross-validation of the remaining clips, and the two FASTT variants are compared with every other method on those
folds.

The configuration is fixed in `fastt.configs.MAIN["librispeech"]` and the code is in `fastt.experiment`. The
features and the stored searches and folds are read from the Hugging Face dataset
[`huseyin-karaca/fastt`](https://huggingface.co/datasets/huseyin-karaca/fastt); nothing is written to it.

| Level | What runs | Needs |
|---|---|---|
| **1** | every table, from the searches and folds stored on the Hub | a CPU, minutes |
| **2** | every fold refitted from the published features (`RETUNE = True`: the search as well) | a CPU, hours |
| **3** | level 2 on features rebuilt from the audio: run `extract` first on the same machine, which writes them to a local folder that level 3 reads (`FASTT_HUB` names another place) | a GPU for `extract` |

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("fastt") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "fastt[paper] @ git+https://github.com/huseyin-karaca/fastt.git@main"], check=True)

In [2]:
from fastt.configs import MAIN
from fastt.experiment import MainExperiment

LEVEL = 1          # 1, 2 or 3 (see above)
RETUNE = False     # levels 2 and 3: run the hyperparameter search again

exp = MainExperiment(MAIN["librispeech"], level=LEVEL, retune=RETUNE)

LibriSpeech: level 1 — search read from the Hub, folds read from the Hub
experts  Parakeet-TDT-0.6b-v2, Canary-Qwen-2.5b, Whisper-large-v3
arms     FASTT-Alternating, FASTT-Boosted, Raw XGBoost (AdaSTT), Raw SDT, SelectKBest XGBoost, SelectKBest SDT, ROVER (weighted), ROVER (confidence-weighted), CN-MBR, Score-Fusion, Hard-Selector (plain CE), Hard-Selector (cost-sensitive CE), Soft-Selector (mixture), QE-Router


## 1. Hyperparameter spaces (Table 2)

The free hyperparameters of every method and their choices; every other parameter is fixed
(`fastt.configs.SPACES`). The spaces are the same on every corpus.

In [3]:
exp.spaces()

Table 2 — hyperparameter spaces
10 random draws per arm; parameters not listed are fixed
  [FASTT-Alternating  (18 free of 18)]
           parameter kind                                   choices / range
      transform_type  cat {nonlinear, linear, diagonal, low_rank, identity}
      num_iterations  cat                                     {6, 12, 3, 1}
        transform_lr  cat                             {0.001, 0.0001, 0.01}
     transform_steps  cat                                    {100, 25, 400}
        surrogate_lr  cat                                {0.01, 0.1, 0.001}
     surrogate_steps  cat                                    {50, 200, 400}
             lambda1  cat                               {0.1, 0.001, 1e-05}
             lambda2  cat                               {1e-05, 0.1, 0.001}
    sel_n_estimators  cat                                             {400}
   sel_learning_rate  cat                            {0.03, 0.3, 0.1, 0.01}
       sel_max_depth  cat                                     {5, 12, 8, 3}
       sel_subsample  cat                                   {1.0, 0.6, 0.8}
       sel_colsample  cat                              {0.6, 1.0, 0.3, 0.1}
sel_min_child_weight  cat                                             {1.0}
      sel_reg_lambda  cat                                             {3.0}
        weight_decay  cat                                            {0.01}
           out_ratio  cat                                   {1.0, 0.1, 0.3}
    bottleneck_ratio  cat                                            {0.05}
  [FASTT-Boosted  (15 free of 15)]
        parameter kind                                   choices / range
   transform_type  cat {identity, low_rank, linear, nonlinear, diagonal}
          lambda1  cat                                           {0.001}
          lambda2  cat                               {0.001, 1e-05, 0.1}
        out_ratio  cat                                             {0.1}
 bottleneck_ratio  cat                                 {0.15, 0.5, 0.05}
       num_rounds  cat                                      {6, 1, 4, 2}
        num_trees  cat                                    {24, 8, 16, 4}
            depth  cat                                               {7}
            lmbda  cat                        {0.01, 0.0001, 1.0, 1e-06}
           lmbda2  cat                                           {1e-07}
          adam_lr  cat                                          {0.0003}
adam_weight_decay  cat                        {0.001, 1e-07, 1e-05, 0.1}
       batch_size  cat                                  {512, 128, 2048}
         patience  cat                                      {40, 20, 10}
   t_weight_decay  cat                                           {1e-06}
  [Raw XGBoost (AdaSTT)  (7 free of 7)]
           parameter kind        choices / range
    sel_n_estimators  cat   {1000, 400, 150, 50}
   sel_learning_rate  cat {0.3, 0.1, 0.03, 0.01}
       sel_max_depth  cat                    {3}
       sel_subsample  cat        {1.0, 0.8, 0.6}
       sel_colsample  cat                  {1.0}
sel_min_child_weight  cat                 {16.0}
      sel_reg_lambda  cat       {3.0, 0.3, 30.0}
  [Raw SDT  (14 free of 15)]
        parameter kind              choices / range
   t_weight_decay  cat                      {1e-06}
          lambda1  cat                      {0.001}
          lambda2  cat                      {0.001}
        out_ratio  cat                        {0.1}
 bottleneck_ratio  cat                       {0.05}
       num_rounds  cat                 {1, 2, 4, 6}
        num_trees  cat                          {8}
            depth  cat                 {2, 7, 5, 3}
            lmbda  cat   {1e-06, 1.0, 0.0001, 0.01}
           lmbda2  cat                        {0.1}
          adam_lr  cat {0.001, 0.003, 0.0003, 0.01}
adam_weight_decay  cat                      {1e-07}
       batch_size  cat             {2048, 512, 128}
         patience  cat              

## 2. Data

The three experts' pooled encoder features, their transcripts, and the transcripts' word-error counters against
the reference, for every clip. A sample of the stored counters is recomputed from the stored transcripts as a
check.

In [4]:
exp.load()

11,126 utterances x 3 routable model(s), from cache .fastt-cache/store_d508f436e94ed336
texts for 3 model(s), from cache
100.0% of 1,200 (utterance, model) counters reproduce exactly


EnsembleStore(librispeech, 'mean', 11,126 utterances, 3 routable)

## 3. Hyperparameter search

A random 25 % of the clips is held out for the search. For every method, ten configurations are drawn at random
from its space (Table 2), each fitted on 70 % of the hold-out and scored on the remaining 30 %; the configuration
with the lowest corpus WER is kept. When several tie on it, the one the paper's run kept is used
(`fastt.configs.SELECTED`), after checking that it is among the tied best. The other 75 % of the clips are never
seen by the search.

In [5]:
exp.search()


=== best fastt_alternating   corpus WER 0.0198   gap closed +5.0%   (10 trials, 0.0 min)
{
  "bottleneck_ratio": 0.05,
  "num_iterations": 1,
  "out_ratio": 0.1,
  "sel_colsample": 1.0,
  "sel_learning_rate": 0.01,
  "sel_max_depth": 3,
  "sel_min_child_weight": 1.0,
  "sel_n_estimators": 400,
  "sel_reg_lambda": 3.0,
  "sel_subsample": 0.8,
  "surrogate_lr": 0.001,
  "surrogate_steps": 50,
  "transform_lr": 0.0001,
  "transform_steps": 400,
  "transform_type": "nonlinear",
  "weight_decay": 0.01
}

=== best fastt_boosted   corpus WER 0.0194   gap closed +9.9%   (10 trials, 0.0 min)
{
  "adam_lr": 0.0003,
  "adam_weight_decay": 1e-07,
  "batch_size": 512,
  "bottleneck_ratio": 0.15,
  "depth": 7,
  "epochs": 200,
  "lmbda": 1e-06,
  "lmbda2": 1e-07,
  "num_rounds": 6,
  "num_trees": 24,
  "out_ratio": 0.1,
  "patience": 20,
  "t_weight_decay": 1e-06,
  "transform_type": "low_rank"
}

=== best adastt   corpus WER 0.0201   gap closed +0.0%   (10 trials, 0.0 min)
{
  "sel_colsample": 1.0


=== best rover_w   corpus WER 0.0194   gap closed +9.9%   (10 trials, 0.0 min)
{
  "alpha": 1.0,
  "conf_source": "cn",
  "null_penalty": 0.5,
  "weight_by_conf": false,
  "weighting": "log_odds"
}

=== best rover_conf   corpus WER 0.0197   gap closed +5.8%   (10 trials, 0.0 min)
{
  "alpha": 0.25,
  "conf_source": "cn",
  "conf_temp": 0.25,
  "null_penalty": 0.25,
  "weight_by_conf": true,
  "weighting": "inverse_wer"
}

=== best cn_mbr   corpus WER 0.0209   gap closed -9.9%   (10 trials, 0.0 min)
{
  "candidates": "members",
  "conf_source": "blend",
  "decode": "hyp",
  "null_prior": 0.25,
  "prior": "inverse_wer",
  "temperature": 10.0,
  "use_conf": true
}

=== best score_fusion   corpus WER 0.0281   gap closed -105.8%   (10 trials, 0.0 min)
{
  "mode": "hard",
  "w_conf": 1.0,
  "w_consensus": -1.0,
  "w_length": -1.0
}

=== best argmin_clf   corpus WER 0.0209   gap closed -10.7%   (10 trials, 0.0 min)
{
  "alpha": 1.0
}



=== best probe   corpus WER 0.0210   gap closed -11.6%   (10 trials, 0.0 min)
{
  "alpha": 1000.0,
  "objective_weight": "length"
}

=== best soft_probe   corpus WER 0.0218   gap closed -22.5%   (10 trials, 0.0 min)
{
  "alpha": 10.0,
  "beta": 0.5,
  "objective_weight": "uniform"
}

=== best qe_router   corpus WER 0.0209   gap closed -9.9%   (10 trials, 0.0 min)
{
  "alpha_decades": 4,
  "alpha_lo": 0.0001,
  "n_alphas": 16,
  "standardize": false,
  "target": "errors"
}
wrote .fastt-cache/runs/level1/hpt_librispeech_795eb630.json  (8 arm(s))


partition,n_utts,share
tune,2782,0.250
eval,8344,0.750
fit,1947,0.175
val,835,0.075


  on the tuning-score rows: bsm(canary_qwen_2_5b) 0.0201   oracle 0.0126   gap 0.0075
  the 8,344 evaluation rows are never seen by a trial
  objectives   : corpus_wer (minimize) — crossval.METRICS['corpus_wer'], on the tuning-score rows
  14 arms x 10 trials, read from the Hub; 8,344 evaluation clips the search never saw


HPTRun('hpt_librispeech_3394aa21', 6 arm(s), best fastt_boosted 0.0194, eval on 8,344 rows)

In [6]:
exp.dataset()

statistic,librispeech
Total clips,11126
Features,3329
Hyperparameter hold-out,2782
Evaluation rows (5x2),8344
Train size (per fold),4172
Test size (per fold),4172
Hours (est.),5.3389
Reference words,212235


In [7]:
exp.selected()

label,trials,hold-out WER,selected configuration
FASTT-Alternating,10,0.0198,"{""bottleneck_ratio"": 0.05, ""num_iterations"": 1, ""out_ratio"": 0.1, ""sel_colsample"": 1.0, ""sel_learning_rate"": 0.01, ""sel_max_depth"": 3, ""sel_min_child_weight"": 1.0, ""sel_n_estimators"": 400, ""sel_reg_lambda"": 3.0, ""sel_subsample"": 0.8, ""surrogate_lr"": 0.001, ""surrogate_steps"": 50, ""transform_lr"": 0.0001, ""transform_steps"": 400, ""transform_type"": ""nonlinear"", ""weight_decay"": 0.01}"
FASTT-Boosted,10,0.0194,"{""adam_lr"": 0.0003, ""adam_weight_decay"": 1e-07, ""batch_size"": 512, ""bottleneck_ratio"": 0.15, ""depth"": 7, ""epochs"": 200, ""lmbda"": 1e-06, ""lmbda2"": 1e-07, ""num_rounds"": 6, ""num_trees"": 24, ""out_ratio"": 0.1, ""patience"": 20, ""t_weight_decay"": 1e-06, ""transform_type"": ""low_rank""}"
Raw XGBoost (AdaSTT),10,0.0201,"{""sel_colsample"": 1.0, ""sel_learning_rate"": 0.03, ""sel_max_depth"": 3, ""sel_min_child_weight"": 16.0, ""sel_n_estimators"": 400, ""sel_reg_lambda"": 0.3, ""sel_subsample"": 0.6}"
Raw SDT,10,0.0201,"{""adam_lr"": 0.001, ""adam_weight_decay"": 1e-07, ""batch_size"": 512, ""depth"": 5, ""epochs"": 200, ""lmbda"": 0.0001, ""lmbda2"": 0.1, ""num_rounds"": 2, ""num_trees"": 8, ""patience"": 20, ""transform_type"": ""identity""}"
SelectKBest XGBoost,10,0.0201,"{""k"": 512, ""sel_colsample"": 1.0, ""sel_learning_rate"": 0.01, ""sel_max_depth"": 3, ""sel_min_child_weight"": 16.0, ""sel_n_estimators"": 1000, ""sel_reg_lambda"": 30.0, ""sel_subsample"": 0.8}"
SelectKBest SDT,10,0.0201,"{""adam_lr"": 0.003, ""adam_weight_decay"": 0.1, ""batch_size"": 2048, ""depth"": 7, ""epochs"": 200, ""k"": 32, ""lmbda"": 1e-06, ""lmbda2"": 0.1, ""num_rounds"": 4, ""num_trees"": 24, ""patience"": 20, ""transform_type"": ""identity""}"
ROVER (weighted),10,0.0194,"{""alpha"": 1.0, ""conf_source"": ""cn"", ""null_penalty"": 0.5, ""weight_by_conf"": false, ""weighting"": ""log_odds""}"
ROVER (confidence-weighted),10,0.0197,"{""alpha"": 0.25, ""conf_source"": ""cn"", ""conf_temp"": 0.25, ""null_penalty"": 0.25, ""weight_by_conf"": true, ""weighting"": ""inverse_wer""}"
CN-MBR,10,0.0209,"{""candidates"": ""members"", ""conf_source"": ""blend"", ""decode"": ""hyp"", ""null_prior"": 0.25, ""prior"": ""inverse_wer"", ""temperature"": 10.0, ""use_conf"": true}"
Score-Fusion,10,0.0281,"{""mode"": ""hard"", ""w_conf"": 1.0, ""w_consensus"": -1.0, ""w_length"": -1.0}"


## 4. 5×2 cross-validation

Five repetitions, each splitting the evaluation clips into two halves with its own seed. Every method is trained
on one half and tested on the other, then the other way round: ten folds, the same for every method.

In [8]:
exp.cross_validate();

  runlog HIT  cv5x2 seed 11 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 22 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 33 — 15/15 group(s) cached, nothing refitted


  runlog HIT  cv5x2 seed 44 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 55 — 15/15 group(s) cached, nothing refitted


5 repetitions x 2 folds x 24 arms, read from the Hub


## 5. Main results (Table 3)

Corpus WER and mean utterance WER (mean ± sd over the ten folds), the share of the oracle gap closed and the
selection accuracy at 0, 1 and 2 words of tolerance, and how often each method picks each expert.

In [9]:
exp.results()

Table 3 — main results, LibriSpeech
mean ± sd over 10 folds; GC / uGC / Acc in %; dist = share of clips routed to Parakeet-TDT-0.6b-v2 / Canary-Qwen-2.5b / Whisper-large-v3; best non-oracle value per column in bold
  [individual]
                                 arm             WER            uWER         GC@0          GC@1           GC@2        uGC@0         uGC@1          uGC@2      Acc@0      Acc@1      Acc@2           dist
                Parakeet-TDT-0.6b-v2 0.0233 ± 0.0004 0.0297 ± 0.0007   -8.8 ± 4.8  -18.4 ± 10.6   -67.2 ± 38.8  -13.1 ± 9.0  -28.5 ± 20.9  -110.5 ± 89.5 86.9 ± 0.4 96.0 ± 0.2 99.5 ± 0.1 1.00-0.00-0.00
                    Canary-Qwen-2.5b 0.0226 ± 0.0005 0.0283 ± 0.0008    0.0 ± 0.0     0.0 ± 0.0      0.0 ± 0.0    0.0 ± 0.0     0.0 ± 0.0      0.0 ± 0.0 88.1 ± 0.4 96.5 ± 0.3 99.4 ± 0.1 0.00-1.00-0.00
                    Whisper-large-v3 0.0295 ± 0.0007 0.0354 ± 0.0009 -81.6 ± 12.6 -167.6 ± 30.4 -609.5 ± 109.3 -63.2 ± 10.9 -133.6 ± 31.0 -514.1 ± 181.3 83.5 ± 0.4 93.7 ± 0.3 98.4 ± 0.2 0.00-0.00-1.00
Best-Single-Model (Canary-Qwen-2.5b) 0.0226 ± 0.0005 0.0283 ± 0.0008    0.0 ± 0.0     0.0 ± 0.0      0.0 ± 0.0    0.0 ± 0.0     0.0 ± 0.0      0.0 ± 0.0 88.1 ± 0.4 96.5 ± 0.3 99.4 ± 0.1 0.00-1.00-0.00
  [external]
                              arm             WER            uWER         GC@0          GC@1           GC@2        uGC@0         uGC@1          uGC@2      Acc@0      Acc@1      Acc@2           dist
                  Random-Ensemble 0.0251 ± 0.0007 0.0309 ± 0.0011  -29.5 ± 6.7  -60.5 ± 14.7  -221.6 ± 63.9  -22.9 ± 6.3  -48.1 ± 14.5  -181.5 ± 63.6 86.2 ± 0.4 95.5 ± 0.3 99.1 ± 0.1 0.33-0.33-0.33
                 ROVER (weighted) 0.0214 ± 0.0003 0.0270 ± 0.0008   13.8 ± 2.2    28.2 ± 3.9   103.1 ± 18.5   11.0 ± 3.3    22.9 ± 6.5    88.5 ± 36.3          —          —          —              —
      ROVER (confidence-weighted) 0.0221 ± 0.0004 0.0283 ± 0.0007    4.6 ± 2.9     9.2 ± 5.7    33.9 ± 21.0   -0.5 ± 5.2   -1.5 ± 11.2    -6.4 ± 44.8          —          —          —              —
                           CN-MBR 0.0230 ± 0.0005 0.0297 ± 0.0009   -5.7 ± 4.6   -11.9 ± 9.8   -43.5 ± 35.7  -12.9 ± 7.9  -27.6 ± 17.9  -105.2 ± 74.5 88.1 ± 0.2 95.9 ± 0.2 99.2 ± 0.1 0.86-0.11-0.03
                     Score-Fusion 0.0303 ± 0.0008 0.0370 ± 0.0009 -90.9 ± 12.0 -186.6 ± 30.7 -679.9 ± 119.6 -77.2 ± 11.5 -163.0 ± 34.7 -624.6 ± 206.6 82.8 ± 0.3 92.9 ± 0.2 98.3 ± 0.2 0.31-0.36-0.34
         Hard-Selector (plain CE) 0.0232 ± 0.0004 0.0293 ± 0.0008   -7.6 ± 4.5   -15.8 ± 9.9   -57.6 ± 35.7   -9.6 ± 9.1  -21.2 ± 20.8   -84.6 ± 89.4 87.1 ± 0.4 96.1 ± 0.3 99.5 ± 0.1 0.99-0.01-0.00
Hard-Selector (cost-sensitive CE) 0.0233 ± 0.0004 0.0297 ± 0.0007   -8.8 ± 4.8  -18.4 ± 10.6   -67.2 ± 38.8  -13.1 ± 9.0  -28.5 ± 20.9  -110.5 ± 89.5 86.9 ± 0.4 96.0 ± 0.2 99.5 ± 0.1 1.00-0.00-0.00
                        QE-Router 0.0237 ± 0.0005 0.0301 ± 0.0011  -13.7 ± 3.2   -28.1 ± 6.8  -102.4 ± 25.3  -16.0 ± 5.8  -33.8 ± 13.0  -133.9 ± 69.5 86.9 ± 0.5 95.7 ± 0.4 99.2 ± 0.1 0.28-0.31-0.41
  [classical]
                 arm             WER            uWER      GC@0      GC@1        GC@2     uGC@0     uGC@1       uGC@2      Acc@0      Acc@1      Acc@2           dist
Raw XGBoost (AdaSTT) 0.0224 ± 0.0006 0.0280 ± 0.0009 1.7 ± 2.3 3.5 ± 4.9 13.0 ± 17.7 2.6 ± 2.9 5.5 ± 6.0 18.9 ± 17.5 88.1 ± 0.4 96.5 ± 0.4 99.4 ± 0.1 0.18-0.82-0.00
             Raw SDT 0.0226 ± 0.0005 0.0283 ± 0.0008 0.0 ± 0.0 0.0 ± 0.0   0.0 ± 0.0 0.0 ± 0.0 0.0 ± 0.0   0.0 ± 0.0 88.1 ± 0.4 96.5 ± 0.3 99.4 ± 0.1 0.00-1.00-0.00
 SelectKBest XGBoost 0.0225 ± 0.0004 0.0281 ± 0.0006 0.3 ± 1.9 0.5 ± 4.1  2.4 ± 15.3 1.0 ± 3.8 1.7 ± 8.4  6.2 ± 34.6 88.0 ± 0.4 96.5 ± 0.2 99.4 ± 0.1 0.31-0.69-0.00
     SelectKBest SDT 0.0226 ± 0.0005 0.0283 ± 0.0008 0.0 ± 0.0 0.0 ± 0.0   0.0 ± 0.0 0.0 ± 0.0 0.0 ± 0.0   0.0 ± 0.0 88.1 ± 0.4 96.5 ± 0.3 99.4 ± 0.1 0.00-1.00-0.00
  [fastt]
              arm             WER            uWER      GC@0       GC@1        GC@2      uGC@0       uGC@1        uGC@2   

In [10]:
exp.routing()

model,n_picks,n_fused,share_parakeet_tdt_0_6b_v2,share_canary_qwen_2_5b,share_whisper_large_v3,top_member,share_top,entropy,entropy_max,share_sd_mean,switch_rate
achievable_oracle,41720,0,0.0035,0.9936,0.0029,canary_qwen_2_5b,0.9936,0.0618,1.5850,0.0006,0.0064
adastt,41720,0,0.1814,0.8185,0.0001,canary_qwen_2_5b,0.8185,0.6847,1.5850,0.0939,0.1815
argmin_clf,41720,0,0.9894,0.0100,0.0006,parakeet_tdt_0_6b_v2,0.9894,0.0884,1.5850,0.0012,0.9900
bsm,41720,0,0.0000,1.0000,0.0000,canary_qwen_2_5b,1.0000,0.0000,1.5850,0.0000,0.0000
canary_qwen_2_5b,41720,0,0.0000,1.0000,0.0000,canary_qwen_2_5b,1.0000,0.0000,1.5850,0.0000,0.0000
cn_mbr,41720,0,0.8608,0.1076,0.0316,parakeet_tdt_0_6b_v2,0.8608,0.6897,1.5850,0.0049,0.8924
fastt_alternating,41720,0,0.3367,0.5895,0.0738,canary_qwen_2_5b,0.5895,1.2559,1.5850,0.1080,0.4105
fastt_boosted,41720,0,0.4188,0.5019,0.0793,canary_qwen_2_5b,0.5019,1.3151,1.5850,0.0671,0.4981
kbest_sdt,41720,0,0.0000,1.0000,0.0000,canary_qwen_2_5b,1.0000,0.0000,1.5850,0.0000,0.0000
kbest_xgboost,41720,0,0.3087,0.6890,0.0023,canary_qwen_2_5b,0.6890,0.9135,1.5850,0.0830,0.3110


## 6. Significance (Table 6)

Every method against the FASTT anchor (the better of the two FASTT variants on corpus WER) on the same ten folds:
Dietterich's 5×2 cv paired $t$ test and Alpaydın's combined 5×2 cv $F$ test, on corpus WER and on mean utterance
WER.

In [11]:
exp.tests()

Table 6 — every arm against the FASTT anchor, LibriSpeech
on the same 10 folds; positive effect = the anchor is better, in the metric's own units; p < 0.05 in bold. The F-test is the one the paper reports.
  [mean utterance WER — anchor FASTT-Boosted]
                       comparator  effect t_dietterich p_dietterich F_alpaydin p_alpaydin
             Parakeet-TDT-0.6b-v2 +0.0018        4.201       0.0085     12.203     0.0064
                 Canary-Qwen-2.5b +0.0004       -0.302       0.7746      0.775     0.6591
                 Whisper-large-v3 +0.0076        6.583       0.0012     51.388     0.0002
                Best-Single-Model +0.0004       -0.302       0.7746      0.775     0.6591
                  Random-Ensemble +0.0030        1.127       0.3110      6.087     0.0299
                 ROVER (weighted) -0.0009       -1.950       0.1086      2.333     0.1811
      ROVER (confidence-weighted) +0.0004        0.681       0.5262      1.677     0.2957
                           CN-MBR +0.0018        2.319       0.0682      7.713     0.0180
                     Score-Fusion +0.0092       11.924       0.0001    147.976     0.0000
         Hard-Selector (plain CE) +0.0014        4.026       0.0101      7.662     0.0182
Hard-Selector (cost-sensitive CE) +0.0018        4.201       0.0085     12.203     0.0064
                        QE-Router +0.0022        0.890       0.4144      3.202     0.1055
             Raw XGBoost (AdaSTT) +0.0001       -0.301       0.7754      0.677     0.7199
                          Raw SDT +0.0004       -0.302       0.7746      0.775     0.6591
              SelectKBest XGBoost +0.0003        0.105       0.9206      0.845     0.6178
                  SelectKBest SDT +0.0004       -0.302       0.7746      0.775     0.6591
                FASTT-Alternating +0.0006        0.936       0.3924      1.123     0.4776
            Oracle (no tolerance) -0.0110      -12.187       0.0001    158.759     0.0000
        Oracle (1 word tolerance) -0.0051       -5.874       0.0020     43.664     0.0003
        Oracle (2 word tolerance) -0.0011       -1.318       0.2447      1.993     0.2312
  [corpus WER — anchor FASTT-Boosted]
                       comparator  effect t_dietterich p_dietterich F_alpaydin p_alpaydin
             Parakeet-TDT-0.6b-v2 +0.0009        7.375       0.0007     33.935     0.0006
                 Canary-Qwen-2.5b +0.0001        0.099       0.9252      0.651     0.7367
                 Whisper-large-v3 +0.0071        5.586       0.0025     38.698     0.0004
                Best-Single-Model +0.0001        0.099       0.9252      0.651     0.7367
                  Random-Ensemble +0.0027        3.801       0.0126     15.927     0.0035
                 ROVER (weighted) -0.0010       -2.820       0.0371      6.602     0.0251
      ROVER (confidence-weighted) -0.0003       -0.740       0.4927      1.591     0.3174
                           CN-MBR +0.0006        1.897       0.1162      3.973     0.0706
                     Score-Fusion +0.0079        7.493       0.0007     59.590     0.0001
         Hard-Selector (plain CE) +0.0008        5.673       0.0024     17.212     0.0029
Hard-Selector (cost-sensitive CE) +0.0009        7.375       0.0007     33.935     0.0006
                        QE-Router +0.0013        1.095       0.3233      3.856     0.0747
             Raw XGBoost (AdaSTT) +0.0000       -0.182       0.8629      0.583     0.7816
                          Raw SDT +0.0001        0.099       0.9252      0.651     0.7367
              SelectKBest XGBoost +0.0001        0.219       0.8350      0.648     0.7389
                  SelectKBest SDT +0.0001        0.099       0.9252      0.651     0.7367
                FASTT-Alternating +0.0001        0.714       0.5074      0.662     0.7297
            Oracle (no tolerance) -0.0084      -20.960       0.0000    444.727     0.0000
        Oracle (1 word tolerance) -0.0040      -11.080       0.0001    133.350     0.0000
        Oracle (2 wo

## 7. Record

The configuration, the selected hyperparameters and every table above, in `main_librispeech.json`;
`manuscript_tables` reads it from the folder `FASTT_RECORDS_DIR` names.

In [12]:
exp.save();

wrote main_librispeech.json
